In [2]:
# Load model (super easy)
import torch
import polars as pl
# model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')  # Large model
# or
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')  # Base model
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Use it
from PIL import Image
import torchvision.transforms as T

transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Using cache found in C:\Users\rdgbr/.cache\torch\hub\facebookresearch_dinov2_main


In [3]:
img = Image.open('images/habitat/coke_can.png')
img_tensor = transform(img).unsqueeze(0)

with torch.no_grad():
    features = model(img_tensor)  # Shape: [1, 1024] for ViT-L
img_tensor.shape, features.shape

(torch.Size([1, 3, 224, 224]), torch.Size([1, 768]))

In [4]:
import polars as pl

habitat_path = 'images/habitat/'
isaac_path = 'images/isaac/'

habitat_df = pl.read_csv('images/habitat/habitat_rr_semantic_anchors.csv')
isaac_df = pl.read_csv('images/isaac/Semantic_anchors.csv')

# Update 'path to image' column and rename columns
isaac_df = isaac_df.with_columns(
    (pl.lit(isaac_path) + pl.col('path to image')).alias('path to image')
).rename({'Object': 'object', 'path to image': 'path'})

habitat_df = habitat_df.with_columns(
    (pl.lit(habitat_path) + pl.col('path to image')).alias('path to image')
).rename({'Object': 'object', 'path to image': 'path'})

print(list(habitat_df['object']))
print(list(isaac_df['object']))

['umbrella', 'soda can', 'red button', 'green button', 'wallet', 'toothbrush', 'lunchbox']
['fire pit', 'camping chair', 'grill', 'teapot', 'silverware', 'coffee maker', 'crumpled cans', 'cooler', 'cardinal', 'coffee bag', 'wall map', 'sneakers', 'formal shoes', 'firewood pile', 'toolbox', 'drill', 'compass', 'ice cubes', 'coke can', 'car key', 'camera', 'sleeping bag']


In [ ]:
# Get embeddings for both environments

image_path_1 = habitat_df.filter(pl.col('object') == 'soda can')['path'][0]
image_path_2 = isaac_df.filter(pl.col('object') == 'coke can')['path'][0]

In [6]:
image_1 = Image.open(image_path_1).convert('RGB')
image_2 = Image.open(image_path_2).convert('RGB')

apartment_tensor = transform(image_1).unsqueeze(0).to(device)
beach_tensor = transform(image_2).unsqueeze(0).to(device)

with torch.no_grad():
    emb_apartment = model(apartment_tensor)  # [1, 1024]
    emb_beach = model(beach_tensor)         # [1, 1024]

# Compute similarity
cosine_sim = torch.nn.functional.cosine_similarity(emb_apartment, emb_beach)
l2_distance = torch.norm(emb_apartment - emb_beach)

print(f"Cosine similarity: {cosine_sim.item():.4f}")
print(f"L2 distance: {l2_distance.item():.4f}")

Cosine similarity: 0.0453
L2 distance: 64.3686
